# DATA 202 Final Project: Does Payroll Buy Wins?
## Notebook 02: Pull the Final 2026 Standings

**Students:** Asher Ring & Elliot Asquith

**What this notebook does:** Downloads the final 2026 regular-season standings with pybaseball, gives each team the same three-letter code Spotrac uses, checks every record against the payroll file, and saves one clean table.

**Reads:** pybaseball (internet connection needed) and `data/mlb_team_payroll_totals.csv`

**Creates:** `data/mlb_standings_2026_final.csv` (30 rows, one per team)

**Slides it supports:** What Each Dataset's Columns Hold (Standings card)

### Download the standings

`standings()` returns a list of six tables, one per division. We stack them into one table of 30 teams.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

DATA_DIR = Path("data")
IMG_DIR = Path("images")
IMG_DIR.mkdir(exist_ok=True)

from pybaseball import standings

YEAR = 2026   # pin the season so this keeps returning 2026 after a new season starts

divisions = standings(YEAR)
print("Division tables:", len(divisions))
standings_raw = pd.concat(divisions, ignore_index=True)
print("Teams:", len(standings_raw))
display(standings_raw.head())

Division tables: 6
Teams: 30


,Tm,W,L,W-L%,GB
0,Tampa Bay Rays,98,64,.605,--
1,New York Yankees,93,68,.578,4.5
2,Boston Red Sox,87,75,.537,11.0
3,Baltimore Orioles,79,82,.491,18.5
4,Toronto Blue Jays,79,83,.488,19.0


### Match team names to Spotrac's team codes

The standings use full names ("Tampa Bay Rays"), while the payroll file uses codes ("TB"). This dictionary lets the two files join on the same `team` column.

In [2]:
TEAM_CODES = {
    "Arizona Diamondbacks": "ARI", "Athletics": "ATH", "Atlanta Braves": "ATL",
    "Baltimore Orioles": "BAL", "Boston Red Sox": "BOS", "Chicago Cubs": "CHC",
    "Chicago White Sox": "CHW", "Cincinnati Reds": "CIN", "Cleveland Guardians": "CLE",
    "Colorado Rockies": "COL", "Detroit Tigers": "DET", "Houston Astros": "HOU",
    "Kansas City Royals": "KC", "Los Angeles Angels": "LAA", "Los Angeles Dodgers": "LAD",
    "Miami Marlins": "MIA", "Milwaukee Brewers": "MIL", "Minnesota Twins": "MIN",
    "New York Mets": "NYM", "New York Yankees": "NYY", "Philadelphia Phillies": "PHI",
    "Pittsburgh Pirates": "PIT", "San Diego Padres": "SD", "San Francisco Giants": "SF",
    "Seattle Mariners": "SEA", "St. Louis Cardinals": "STL", "Tampa Bay Rays": "TB",
    "Texas Rangers": "TEX", "Toronto Blue Jays": "TOR", "Washington Nationals": "WSH",
}

standings_df = pd.DataFrame({
    "team": standings_raw["Tm"].map(TEAM_CODES),
    "team_name": standings_raw["Tm"],
    "wins": standings_raw["W"].astype(int),
    "losses": standings_raw["L"].astype(int),
})
standings_df["games_played"] = standings_df["wins"] + standings_df["losses"]
standings_df["win_pct"] = (standings_df["wins"] / standings_df["games_played"]).round(3)
standings_df = standings_df.sort_values("win_pct", ascending=False).reset_index(drop=True)

# Every team name must have found a code
assert standings_df["team"].notna().all(), standings_df[standings_df["team"].isna()]
assert standings_df["team"].nunique() == 30
display(standings_df)

,team,team_name,wins,losses,games_played,win_pct
0,MIL,Milwaukee Brewers,103,59,162,0.636
1,LAD,Los Angeles Dodgers,100,62,162,0.617
2,TB,Tampa Bay Rays,98,64,162,0.605
3,ATL,Atlanta Braves,94,68,162,0.580
4,NYY,New York Yankees,93,68,161,0.578
5,SD,San Diego Padres,91,71,162,0.562
6,CHC,Chicago Cubs,89,73,162,0.549
7,PHI,Philadelphia Phillies,88,74,162,0.543
8,BOS,Boston Red Sox,87,75,162,0.537
9,ARI,Arizona Diamondbacks,86,76,162,0.531


### Check the records against the payroll file

Spotrac lists each team's record too. Two independent sources agreeing on all 30 records is a good sign both are right.

In [3]:
teams = pd.read_csv(DATA_DIR / "mlb_team_payroll_totals.csv")
check = teams[["team", "wins", "losses"]].merge(standings_df, on="team", how="outer",
                                              suffixes=("_spotrac", "_standings"), indicator=True)
print(check["_merge"].value_counts())
mismatch = check[(check["wins_spotrac"] != check["wins_standings"]) | (check["losses_spotrac"] != check["losses_standings"])]
print("Teams whose records don't match:", len(mismatch))
assert len(mismatch) == 0
print("All 30 records match.")
print("Teams that played 161 games:", list(standings_df.loc[standings_df["games_played"] == 161, "team"]))

_merge
both          30
left_only      0
right_only     0
Name: count, dtype: int64
Teams whose records don't match: 0
All 30 records match.
Teams that played 161 games: ['NYY', 'BAL']


### Save

In [4]:
standings_df.to_csv(DATA_DIR / "mlb_standings_2026_final.csv", index=False)
print("Saved", DATA_DIR / "mlb_standings_2026_final.csv")

Saved data/mlb_standings_2026_final.csv
